In [37]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.linear_model import LinearRegression, BayesianRidge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
import shap

In [38]:
import warnings
warnings.filterwarnings("ignore")

In [39]:
def trans_donnees(X):
    X1 = X.copy()
    
    # Suppression des variables avec trop de valeurs manquantes (>50%)
    X1.drop(columns=[
        'Numero_Conteneur_Entree', 'Salle_Balance_Entree', 'Region_Entree', 'IP_Entree',
        'Numero_Conteneur_Sortie', 'ID_Balance_Sortie', 'Region_Sortie', 'IP_Sortie',
        'Routine_Manutention', 'Numero_Sequence', 'Remarque', 'Annule', 'Annule_Par', 'Date_Annulation'
    ], inplace=True, errors='ignore')
    
    # Suppression des identifiants inutiles (aucune valeur prédictive)
    X1.drop(columns=[
        'Numero_Camion', 'Plaque_Camion', 'ID_Cargaison',
        'Connaissement', 'Numero_Marque', 'Numero_Pesee', 'Voyage', 'Statut'
    ], inplace=True, errors='ignore')
    
    # Suppression des doublons
    X1 = X1.drop_duplicates()
    
    # Suppression valeurs aberrantes (variables quantitatives)
    X1.loc[X1['Poids_Brut_Camion_kg'] > 100000, 'Poids_Brut_Camion_kg'] = np.nan
    X1.loc[X1['Poids_Cargaison_kg'] > 100000, 'Poids_Cargaison_kg'] = np.nan
    X1.loc[X1['Poids_Camion_Sortie_kg'] > 100000, 'Poids_Camion_Sortie_kg'] = np.nan
    X1.loc[X1['Poids_Tare_kg'] > 100000, 'Poids_Tare_kg'] = np.nan
    X1.loc[X1['Poids_Camion_Entree_kg'] > 100000, 'Poids_Camion_Entree_kg'] = np.nan
   # Par ceci — seuil à 10 000 min (~7 jours, plus réaliste) :
    X1.loc[X1['Surestarie_min'] > 10000, 'Surestarie_min'] = np.nan
    X1.loc[X1['Surestarie_min'] < 0, 'Surestarie_min'] = np.nan
    
    # Suppression variable redondante
    if 'Poids_Brut_Camion_kg' in X1.columns:
        X1.drop(columns=['Poids_Brut_Camion_kg'], inplace=True)
    
    # Suppression des lignes sans variable cible
    X1.dropna(subset=['Surestarie_min'], inplace=True)
    
    # Transformation logarithmique
    log_vars = ['Surestarie_min', 'Poids_Tare_kg',
                'Poids_Cargaison_kg', 'Poids_Camion_Sortie_kg']
    for var in log_vars:
        X1[f'log_{var}'] = np.log1p(X1[var])
    
    # Suppression du log de la cible (pas une feature)
    X1.drop(columns=['log_Surestarie_min'], inplace=True)

    # Feature engineering temporel
    X1['Heure_Entree_Gate'] = pd.to_datetime(X1['Heure_Entree_Gate'], format='%d/%m/%Y %H:%M')
    X1['Heure_Sortie_Gate'] = pd.to_datetime(X1['Heure_Sortie_Gate'], format='%d/%m/%Y %H:%M')
    
    X1['heure_entree'] = X1['Heure_Entree_Gate'].dt.hour
    X1['jour_semaine'] = X1['Heure_Entree_Gate'].dt.dayofweek
    X1['mois']         = X1['Heure_Entree_Gate'].dt.month
    
    X1.drop(columns=['Heure_Entree_Gate', 'Heure_Sortie_Gate'], inplace=True)
    
    # Typage des variables qualitatives
    var_quali = ['Nom_Cargaison', 'Nom_Navire', 'Operateur_Entree', 
                 'Operateur_Sortie', 'Type_Travail']
    for var in var_quali:
        X1[var] = X1[var].astype('object')
    
    return X1

In [40]:
train = pd.read_csv('C:/Users/LENOVO/Documents/Stage_DMP/port-ai-project/data/train.csv', index_col=0)
print(train.shape)
print(train.head())
print(train.columns)

(25498, 34)
         Numero_Camion Plaque_Camion ID_Cargaison   Connaissement  \
Statut                                                              
Gate-out    WDJB911D61     DJB911D61  DC240305005               2   
Gate-out  VCH042677500   CH042677500  DC240219002       TABUK0001   
Gate-out  VCH043848864   CH043848864  DC230909091   NSSCB23002351   
Gate-out     WETA18383      ETA18383  DC240224088  HK23170XDBT016   
Gate-out     WETA22810      ETA22810  DC240505007  NJ2401LYGDJ202   

                                   Numero_Marque  Nom_Cargaison   Nom_Navire  \
Statut                                                                         
Gate-out                      MILLINGWHEATINBULK          Wheat  BOS BOUTROS   
Gate-out                       JTMAA7BJ4P4063170        Vehicle  BAHRI TABUK   
Gate-out                       RKLKABAG3P0509459        Vehicle      JIGJIGA   
Gate-out  CCCC-FHECETHIOPIAEAHPROJECTVIADJIBOUTI         *Rebar     HELENA K   
Gate-out          BELTCO

In [41]:
train = trans_donnees(train)
X_train = train.drop(['Surestarie_min'], axis=1)
y_train = train['Surestarie_min']

In [42]:
val = pd.read_csv('C:/Users/LENOVO/Documents/Stage_DMP/port-ai-project/data/validation.csv', index_col=0)
print(val.head())

         Numero_Camion Plaque_Camion ID_Cargaison   Connaissement  \
Statut                                                              
Gate-out    WDJB288D62     DJB288D62  DC240214010   SSMCB24000009   
Gate-out      WET88021       ET88021  DC240114020   SSMCB24000002   
Gate-out  VCH042736646   CH042736646  DC240224043  HK23170LDBT044   
Gate-out     WETA10211      ETA10211  DC240123018   SSMCB24000005   
Gate-out     WET526D67      ET526D67  DC231121003        MZDDJI01   

                             Numero_Marque Nom_Cargaison    Nom_Navire Voyage  \
Statut                                                                          
Gate-out                    NPSBFERTILIZER    Fertilizer        ZURICH  12024   
Gate-out                    NPSBFERTILIZER    Fertilizer        SABAEK  04L23   
Gate-out                 LZZ5ELSC8PN256285       Vehicle      HELENA K  23170   
Gate-out                     NPSFERTILIZER    Fertilizer       CORINNA     49   
Gate-out  NONALLOYSTEELBILLETS

In [43]:
val = trans_donnees(val)
X_val = val.drop(['Surestarie_min'], axis=1)
y_val = val['Surestarie_min']

In [44]:
#création des différents jeux de données avec utilisation d'un pipeline
var_quali = ['Nom_Cargaison', 'Nom_Navire', 'Operateur_Entree', 
             'Operateur_Sortie', 'Type_Travail']

# Ajout des features temporelles et log
var_quanti = ['Poids_Tare_kg', 'Poids_Cargaison_kg', 
              'Poids_Camion_Entree_kg', 'Poids_Camion_Sortie_kg',
              'heure_entree', 'jour_semaine', 'mois',
              'log_Poids_Tare_kg', 'log_Poids_Cargaison_kg',
              'log_Poids_Camion_Sortie_kg']
#sans Surestarie_min puisque c'est la variable à prédire

var_moy = ['Poids_Tare_kg', 'Poids_Camion_Entree_kg',
           'heure_entree', 'jour_semaine', 'mois']
var_med = ['Poids_Cargaison_kg', 'Poids_Camion_Sortie_kg',
           'log_Poids_Tare_kg', 'log_Poids_Cargaison_kg',
           'log_Poids_Camion_Sortie_kg']

### Imputation Simple

In [45]:
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])#permet d'enchainer les deux sur la même variable
preprocessor_simple = ColumnTransformer([#chaque ligne est indpendante, impératif qu'une variable ne passe que dans une ligne
    ('num_med', SimpleImputer(strategy="median"), var_med),
    ('num_moy', SimpleImputer(strategy="mean"), var_moy),
    ('cat', cat_pipeline, var_quali)#comme deux lignes sur les quali=> un pipeline qui englobe les deux lignes
])
pipe_simple_arbre = Pipeline([
    ('comp', preprocessor_simple),
    ('arbre', DecisionTreeRegressor())
])

In [46]:
#Entrainement
pipe_simple_arbre.fit(X_train, y_train)
#prediction sur train
y_pred_train = pipe_simple_arbre.predict(X_train)
#prediction sur test
y_pred_val = pipe_simple_arbre.predict(X_val)

In [47]:
def qualite_reg(y, y_pred):
    mse = mean_squared_error(y, y_pred)
    print("MSE:", mse)
    print("RMSE", np.sqrt(mse))
    print("R²:", r2_score(y, y_pred))
    print("MAE", mean_absolute_error(y, y_pred))

In [48]:
#résultat obtenus en apprentissage
print("apprentissage")
qualite_reg(y_train, y_pred_train)
#résultat obtenus en validation
print('validation')
qualite_reg(y_val, y_pred_val)
#très gros surapprentissage!
#important d'ajouter des hyperparamètres

apprentissage
MSE: 24635.619223140537
RMSE 156.9573802761136
R²: 0.9695271653514483
MAE 18.657802672455947
validation
MSE: 449747.81812511606
RMSE 670.6324016367805
R²: 0.3775247819738521
MAE 281.93756367963164


In [14]:
param_grid = { 
    'arbre__max_depth': [3, 5, 7, 10], #bien pensé que le nom indique avant les __ correspond au nom ds le pipeline qui fait ref à la méthode
    'arbre__min_samples_split': [25, 50, 35]} 

In [15]:
grid = GridSearchCV( 
    estimator=pipe_simple_arbre, 
    param_grid=param_grid, 
    cv=5, 
    scoring='neg_mean_squared_error', 
    n_jobs=-1 )
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('comp',
                                        ColumnTransformer(transformers=[('num_med',
                                                                         SimpleImputer(strategy='median'),
                                                                         ['Poids_Cargaison_kg',
                                                                          'Poids_Camion_Sortie_kg',
                                                                          'log_Poids_Tare_kg',
                                                                          'log_Poids_Cargaison_kg',
                                                                          'log_Poids_Camion_Sortie_kg']),
                                                                        ('num_moy',
                                                                         SimpleImputer(),
                                                                         ['Poids_Tare_kg',
                                                                          'Poids_Camion_Entree_kg',
                                                                          'heure_entree',
                                                                          'jour_semaine',...
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(drop='first',
                                                                                                        handle_unknown='ignore'))]),
                                                                         ['Nom_Cargaison',
                                                                          'Nom_Navire',
                                                                          'Operateur_Entree',
                                                                          'Operateur_Sortie',
                                                                          'Type_Travail'])])),
                                       ('arbre', DecisionTreeRegressor())]),
             n_jobs=-1,
             param_grid={'arbre__max_depth': [3, 5, 7, 10],
                         'arbre__min_samples_split': [25, 50, 35]},
             scoring='neg_mean_squared_error')

In [16]:
print("Meilleurs paramètres :", grid.best_params_) 
print("RMSE :", np.sqrt(mean_squared_error(y_val, grid.predict(X_val))))

Meilleurs paramètres : {'arbre__max_depth': 10, 'arbre__min_samples_split': 25}
RMSE : 1253.3029129400697


In [17]:
#résultat obtenus en apprentissage
print("apprentissage")
y_pred_train = grid.predict(X_train)
qualite_reg(y_train, y_pred_train)
#résultat obtenus en validation
print('validation')
y_pred_val = grid.predict(X_val)
qualite_reg(y_val, y_pred_val)

apprentissage
MSE: 1186681.554496
RMSE 1089.349142605804
R²: 0.6252714728571588
MAE 490.79858099380886
validation
MSE: 1570768.191584064
RMSE 1253.3029129400697
R²: 0.17781025335757783
MAE 498.4936142798864


### Imputation KNN

In [18]:
#avec une imputation KNN
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])#permet d'enchainer les deux sur la même variable
preprocessor_simple = ColumnTransformer([#chaque ligne est indpendante, impératif qu'une variable ne passe que dans une ligne
    ('num_KNN', KNNImputer(n_neighbors=20), var_quanti),
    ('cat', cat_pipeline, var_quali)#comme deux lignes sur les quali=> un pipeline qui englobe les deux lignes
])
pipe_KNN_arbre = Pipeline([
    ('comp', preprocessor_simple),
    ('arbre', DecisionTreeRegressor())
])

In [19]:
#Entrainement
pipe_KNN_arbre.fit(X_train, y_train)
#prediction sur train
y_pred_train = pipe_KNN_arbre.predict(X_train)
#prediction sur test
y_pred_val = pipe_KNN_arbre.predict(X_val)

In [20]:
#résultat obtenus en apprentissage
print("apprentissage")
qualite_reg(y_train, y_pred_train)
#résultat obtenus en validation
print('validation')
qualite_reg(y_val, y_pred_val)

apprentissage
MSE: 30843.03316549848
RMSE 175.6218470620853
R²: 0.9902604330985546
MAE 19.66892675237319
validation
MSE: 732746.2065089104
RMSE 856.0059617250982
R²: 0.6164574625901986
MAE 308.65685214323435


In [49]:
param_grid = { 
    'arbre__max_depth': [3, 5, 7, 10, 15], #bien pensé que le nom indique avant les __ correspond au nom ds le pipeline qui fait ref à la méthode
    'arbre__min_samples_split': [25, 50, 15]} 

In [50]:
grid = GridSearchCV( 
    estimator=pipe_KNN_arbre, 
    param_grid=param_grid, 
    cv=5, 
    scoring='neg_mean_squared_error', 
    n_jobs=-1 )
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('comp',
                                        ColumnTransformer(transformers=[('num_KNN',
                                                                         KNNImputer(n_neighbors=20),
                                                                         ['Poids_Tare_kg',
                                                                          'Poids_Cargaison_kg',
                                                                          'Poids_Camion_Entree_kg',
                                                                          'Poids_Camion_Sortie_kg',
                                                                          'heure_entree',
                                                                          'jour_semaine',
                                                                          'mois',
                                                                          'log_Poids_Tare_kg',
                                                                          'log_Poids_Cargaison_kg',
                                                                          'log_Poids_Camion_Sortie_kg']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('im...
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(drop='first',
                                                                                                        handle_unknown='ignore'))]),
                                                                         ['Nom_Cargaison',
                                                                          'Nom_Navire',
                                                                          'Operateur_Entree',
                                                                          'Operateur_Sortie',
                                                                          'Type_Travail'])])),
                                       ('arbre', DecisionTreeRegressor())]),
             n_jobs=-1,
             param_grid={'arbre__max_depth': [3, 5, 7, 10, 15],
                         'arbre__min_samples_split': [25, 50, 15]},
             scoring='neg_mean_squared_error')

In [51]:
print("Meilleurs paramètres :", grid.best_params_) 
print("RMSE :", np.sqrt(mean_squared_error(y_val, grid.predict(X_val))))

Meilleurs paramètres : {'arbre__max_depth': 15, 'arbre__min_samples_split': 15}
RMSE : 625.7674817225469


In [52]:
#résultat obtenus en apprentissage
print("apprentissage")
y_pred_train = grid.predict(X_train)
qualite_reg(y_train, y_pred_train)
#résultat obtenus en validation
print('validation')
y_pred_val = grid.predict(X_val)
qualite_reg(y_val, y_pred_val)

apprentissage
MSE: 355253.97152978834
RMSE 596.0318544589612
R²: 0.560571405386071
MAE 343.8063330132377
validation
MSE: 391584.9411813781
RMSE 625.7674817225469
R²: 0.45802533816890045
MAE 369.6977710647908


### Imputation Iteratives

In [53]:
#Avec une imputation iterative : bayesianRidge
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])#permet d'enchainer les deux sur la même variable
preprocessor_simple = ColumnTransformer([#chaque ligne est indpendante, impératif qu'une variable ne passe que dans une ligne
    ('num_BR', IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=2026), var_quanti),
    ('cat', cat_pipeline, var_quali)#comme deux lignes sur les quali=> un pipeline qui englobe les deux lignes
])
pipe_IIBR_arbre = Pipeline([
    ('comp', preprocessor_simple),
    ('arbre', DecisionTreeRegressor())
])

In [54]:
#Entrainement
pipe_IIBR_arbre.fit(X_train, y_train)
#prediction sur train
y_pred_train = pipe_IIBR_arbre.predict(X_train)
#prediction sur test
y_pred_val = pipe_IIBR_arbre.predict(X_val)

In [55]:
#résultat obtenus en apprentissage
print("apprentissage")
qualite_reg(y_train, y_pred_train)
#résultat obtenus en validation
print('validation')
qualite_reg(y_val, y_pred_val)

apprentissage
MSE: 24635.619223140537
RMSE 156.9573802761136
R²: 0.9695271653514483
MAE 18.657802672455947
validation
MSE: 477406.54073659924
RMSE 690.9461199953288
R²: 0.33924361929988855
MAE 294.3729189506709


In [56]:
param_grid = { 
    'arbre__max_depth': [3, 5, 7, 10, 15], #bien pensé que le nom indique avant les __ correspond au nom ds le pipeline qui fait ref à la méthode
    'arbre__min_samples_split': [30, 50, 15, 25]} 
grid = GridSearchCV( 
    estimator=pipe_IIBR_arbre, 
    param_grid=param_grid, 
    cv=5, 
    scoring='neg_mean_squared_error', 
    n_jobs=-1 )
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('comp',
                                        ColumnTransformer(transformers=[('num_BR',
                                                                         IterativeImputer(estimator=BayesianRidge(),
                                                                                          random_state=2026),
                                                                         ['Poids_Tare_kg',
                                                                          'Poids_Cargaison_kg',
                                                                          'Poids_Camion_Entree_kg',
                                                                          'Poids_Camion_Sortie_kg',
                                                                          'heure_entree',
                                                                          'jour_semaine',
                                                                          'mois',
                                                                          'log_Poids_Tare_kg',
                                                                          'log_Poids_Cargaison_kg',
                                                                          'log_Poids_Camion_Sortie...
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(drop='first',
                                                                                                        handle_unknown='ignore'))]),
                                                                         ['Nom_Cargaison',
                                                                          'Nom_Navire',
                                                                          'Operateur_Entree',
                                                                          'Operateur_Sortie',
                                                                          'Type_Travail'])])),
                                       ('arbre', DecisionTreeRegressor())]),
             n_jobs=-1,
             param_grid={'arbre__max_depth': [3, 5, 7, 10, 15],
                         'arbre__min_samples_split': [30, 50, 15, 25]},
             scoring='neg_mean_squared_error')

In [57]:
print("Meilleurs paramètres :", grid.best_params_) 
print("RMSE :", np.sqrt(mean_squared_error(y_val, grid.predict(X_val))))

Meilleurs paramètres : {'arbre__max_depth': 15, 'arbre__min_samples_split': 15}
RMSE : 661.2350579310987


In [58]:
#résultat obtenus en apprentissage
print("apprentissage")
y_pred_train = grid.predict(X_train)
qualite_reg(y_train, y_pred_train)
#résultat obtenus en validation
print('validation')
y_pred_val = grid.predict(X_val)
qualite_reg(y_val, y_pred_val)

apprentissage
MSE: 380202.6137026098
RMSE 616.6057198101635
R²: 0.5297113794718795
MAE 358.3836912732585
validation
MSE: 437231.8018371435
RMSE 661.2350579310987
R²: 0.39484762302765164
MAE 387.0945445560147


In [ ]:
#meilleur résultat compromis : à compléter après résultats

In [62]:
# =========================
# Meilleur modèle retenu : Arbre + KNN (sans tuning)

# =========================

#avec une imputation KNN
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])#permet d'enchainer les deux sur la même variable
preprocessor_simple = ColumnTransformer([#chaque ligne est indpendante, impératif qu'une variable ne passe que dans une ligne
    ('num_KNN', KNNImputer(n_neighbors=20), var_quanti),
    ('cat', cat_pipeline, var_quali)#comme deux lignes sur les quali=> un pipeline qui englobe les deux lignes
])
pipe_KNN_arbre = Pipeline([
    ('comp', preprocessor_simple),
    ('arbre', DecisionTreeRegressor())
])


In [ ]:
import pandas as pd
import shap

# =========================
# Meilleur modèle retenu : Arbre + Imputation KNN (sans tuning)

# =========================
pipe_KNN_arbre = Pipeline([
    ('comp', preprocessor_simple),
    ('arbre', DecisionTreeRegressor())
])

pipe_KNN_arbre.fit(X_train, y_train)

print("Apprentissage")
y_pred_train = pipe_KNN_arbre.predict(X_train)
qualite_reg(y_train, y_pred_train)

print("Validation")
y_pred_val = pipe_KNN_arbre.predict(X_val)
qualite_reg(y_val, y_pred_val)

# =========================
# Transformation des données
# =========================
preprocess = pipe_KNN_arbre.named_steps["comp"]
X_train_trans = preprocess.transform(X_train)
if hasattr(X_train_trans, "toarray"):
    X_train_trans = X_train_trans.toarray()

feature_names = preprocess.get_feature_names_out()
X_train_trans = pd.DataFrame(
    X_train_trans,
    columns=feature_names,
    index=X_train.index
)

# =========================
# SHAP — Importance des variables
# =========================
model = pipe_KNN_arbre.named_steps["arbre"]
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_trans)

shap.summary_plot(
    shap_values,
    X_train_trans,
    feature_names=X_train_trans.columns,
    plot_type="bar"
)



Apprentissage
MSE: 24635.619223140537
RMSE 156.9573802761136
R²: 0.9695271653514483
MAE 18.657802672455947
Validation
MSE: 430590.24057197926
RMSE 656.1937523109918
R²: 0.4040399017446482
MAE 289.94871982956136


### Conclusion 

In [ ]:
# =========================
# Conclusion Arbre de Décision
# =========================
# Meilleure imputation : KNN sans hyperparamétrage
# RMSE val. = 856 min — R² = 0.616

# Surapprentissage massif sur toutes les imputations :
# - Simple  : R² train = 0.970 vs R² val. = 0.378 (RMSE train=157 vs val=671)
# - KNN     : R² train = 0.990 vs R² val. = 0.616 (RMSE train=176 vs val=856)
# - Itérat. : R² train = 0.970 vs R² val. = 0.339 (RMSE train=157 vs val=691)

# Le tuning n'améliore pas les résultats — il les dégrade :
# - Simple  : R² val. 0.378 → 0.178 après hyperparamétrage
# - KNN     : R² val. 0.616 → 0.458 après hyperparamétrage
# - Itérat. : R² val. 0.339 → 0.395 (légère amélioration, reste insuffisant)

# → L'arbre de décision est intrinsèquement instable sur ces données :
#    il mémorise le jeu d'entraînement sans généraliser.
# → La contrainte de profondeur (max_depth, min_samples_split) réduit
#    le surapprentissage mais détériore aussi la capacité prédictive.
# → Solution : passer à la Forêt Aléatoire, qui corrige ce problème
#    en moyennant des centaines d'arbres entraînés sur des sous-échantillons
#    aléatoires, réduisant la variance sans sacrifier la précision.